In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sc.settings.verbosity = 3 
sc.logging.print_header()
sc.settings.set_figure_params(dpi=300, transparent = True, format = 'pdf', vector_friendly = True)

In [ ]:
figure = "Figure_sup_doublets"

In [ ]:
sc.settings.figdir = './Figure_plots/'+figure

In [ ]:
umap_cmap = sns.blend_palette(['xkcd:light grey', 'xkcd:blueberry'], as_cmap = True)

# Declaring the input files

In [ ]:
adata = sc.read_h5ad('./h5ad/analysis_250528_f/Smed_L78-L47_20250523_doublets_processed.h5ad')

In [ ]:
adata

In [ ]:
clusteringlayer = 'leiden_2.5'

In [ ]:
# numbers

In [ ]:
# number of doublets by experiment
adata.obs[adata.obs['predicted_doublet'] == True].groupby('Experiment').describe()

# Plots

## umaps

In [ ]:
# check the doublet score
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='doublet_score', size = 20, color_map = umap_cmap, frameon=False, save= figure + '_doublet-score.pdf' )

In [ ]:
adata.uns['predicted_doublet_colors'] = ['#add8e6', '#CD1C18']

In [ ]:
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='predicted_doublet',  size = 20, color_map = umap_cmap, frameon=False, title = 'Predicted doublets',
               save = figure + '_predicted_doublets')

In [ ]:
with plt.rc_context({'figure.figsize': (15, 15)}):
    sc.pl.umap(adata, color=clusteringlayer,  size = 50, color_map = umap_cmap, frameon=False, legend_loc= 'on data', title = 'Leiden clusters',
               save = figure + '_clusters2.5')

## barplots

In [ ]:
# mean doublet score per cluster
mean_doublet = adata.obs.groupby('leiden_2.5').describe()['doublet_score']['mean']

In [ ]:
ax = mean_doublet.plot( kind='bar', width=0.9, color = 'black')

ax.grid(axis='y', linestyle='--', linewidth=0.8, alpha=0.7)  # horizontal lines
ax.grid(axis='x', visible=False)  # hide vertical lines
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')

plt.gcf().set_size_inches(30, 5)

plt.savefig('./Figure_plots/'+figure + '/'+ figure + '_mean-doublets.pdf')
plt.show()

In [ ]:
# percentages of doublets per cluster
tot_cluster = adata.obs['leiden_2.5'].value_counts()
doublet_cluster = adata.obs[adata.obs['predicted_doublet'] == True]['leiden_2.5'].value_counts()
perc_cluster = (doublet_cluster / tot_cluster)*100 

In [ ]:
ax = perc_cluster.plot( kind='bar', width=0.9, color = 'black')

ax.grid(axis='y', linestyle='--', linewidth=0.8, alpha=0.7)  # horizontal lines
ax.grid(axis='x', visible=False)  # hide vertical lines
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='right')

plt.gcf().set_size_inches(30, 5)

plt.savefig('./Figure_plots/'+figure + '/'+ figure + '_perc-doublets.pdf')
plt.show()

In [ ]:
# clusters with more than 10% of doublets
thr_cluster = 10
li_doublets = list(perc_cluster[perc_cluster > thr_cluster].index)

In [ ]:
with plt.rc_context({'figure.figsize': (10, 10)}):
    sc.pl.umap(adata, color='leiden_2.5', groups = li_doublets,  size = 20, color_map = umap_cmap, frameon=False,  
               save = figure + '_doublets-leiden2.5')